In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_analog_emulator.interpreter import Interpreter

printer = Post(PrettyPrint())

source = """
a=2 \n b = a + 3 \n H = %I %* %X \n c = true \n d = not c \n 
e = c and d \n f = c or d \n g = a <= b \n h = a >= b \n 
i = a == b \n k = a != b \n l = a - 1 \n m = a * b \n n = 2^3 \n
"""

source = """ 
a = 2 \n b = 5 \n 
if (a > 0) { \n b = 3 } \n
if (b < 0) { \n a = 5} \n
else { \n a = 10}
if (a < 0) { \n b = 10}
"""

source = """ 
n = 5 \n
while (n > 0) { \n n = n - 1}
"""

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(circuit=circuit, cfg=cfg, symbol_table=symbol_table)


In [2]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'n',
   'value': {'class_': 'MathNum', 'value': 5}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'branch',
  'stmt': {'class_': 'BoolGreaterThan',
   'expr1': {'class_': 'Access', 'name': 'n'},
   'expr2': {'class_': 'MathNum', 'value': 0}},
  'preds': [1, 3],
  'succs': [3, 4],
  'exit_nodes': [],
  'edge_labels': {3: 'true', 4: 'false'}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'n',
   'value': {'class_': 'MathAdd',
    'expr1': {'class_': 'MathNum', 'value': -1},
    'expr2': {'class_': 'Access', 'name': 'n'}}},
  'preds': [2],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stop',
  'stmt': {},
  'preds': [2],
  'suc

In [3]:
interpreter = Interpreter(graph=cfg)
interpreter.run()
interpreter.status()

{'n': 0}